Author: Giovanni Squillero <[giovanni.squillero@polito.it](giovanni.squillero@polito.it)>  
Copyright © 2026 [Giovanni Squillero](https://squillero.github.io/) / [Politecnico di Torino](https://www.polito.it)  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE) for details.  

![](i/susi1.webp)

Place the four signs in front of the correct doors and determine her hut.

| Italian                                          | English                                 |
| ------------------------------------------------ | --------------------------------------- |
| Gianni, qual è la nostra cabina?                 | Gianni, which is our beach hut?         |
| Questi quattro cartelli ti aiuteranno a trovarla | These four signs will help you find it. |
| Cartello 1: NON È LA 3                           | Sign 1: It is not #3                    |
| Cartello 2: È ACCANTO A QUESTA                   | Sign 2: It is next to this one          |
| Cartello 3: È ACCANTO A QUESTA                   | Sign 3: It is next to this one          |
| Cartello 4: NON È ACCANTO A QUESTA               | Sign 4: It is not next to this one      |

**Note**: In this context, *cabina* refers to a beach changing hut/cabin commonly found at Italian beaches.

In [1]:
import logging
from itertools import permutations
from icecream import ic

In [2]:
# Both the RiSEs' z3 binary and the z3-solver Python package are needed.
# see: https://en.wikipedia.org/wiki/Z3_Theorem_Prover
import z3

In [3]:
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("Susi")

## Use Python to determine the position of the signs, then the solver for `hut`

In [4]:
for sign_pos in permutations(range(1, 4 + 1)):
    solver = z3.Solver()
    hut = z3.Int("hut")  # What about a EnumSort?
    solver.add(hut >= 1, hut <= 4)  # can't do 1 <= hut <= 4 :/
    # 1. IT IS NOT #3
    assertion = hut != 3
    solver.add(assertion)
    log.debug(f'Assertions: On hut #{sign_pos[0]}\'s door: "It is not hut #3" // {assertion}')
    # 2. IT IS NEXT TO THIS ONE
    assertion = z3.Or(hut == sign_pos[1] - 1, hut == sign_pos[1] + 1)
    solver.add(assertion)
    log.debug(f"Assertions: It is next to hut #{sign_pos[1]} // {assertion}")
    # 3. IT IS NEXT TO THIS ONE
    assertion = z3.Or(hut == sign_pos[2] - 1, hut == sign_pos[2] + 1)
    solver.add(assertion)
    log.debug(f"Assertions: It is next to hut #{sign_pos[2]} // {assertion}")
    # 4. IT IS NOT NEXT TO THIS
    assertion = z3.And(hut != sign_pos[3] - 1, hut != sign_pos[3] + 1)
    solver.add(assertion)
    log.debug(f"Assertions: It is not next to hut #{sign_pos[3]} // {assertion}")
    solver.add()

    if solver.check() == z3.sat:
        ic(sign_pos, solver.assertions(), solver.model())
    log.debug("")

ic| sign_pos: (2, 1, 3, 4)
    solver.assertions(): [hut >= 1,
                          hut <= 4,
                          hut != 3,
                          Or(hut == 0, hut == 2),
                          Or(hut == 2, hut == 4),
                          And(hut != 3, hut != 5)]
    solver.model(): [hut = 2]
ic| sign_pos: (2, 3, 1, 4)
    solver.assertions(): [hut >= 1,
                          hut <= 4,
                          hut != 3,
                          Or(hut == 2, hut == 4),
                          Or(hut == 0, hut == 2),
                          And(hut != 3, hut != 5)]
    solver.model(): [hut = 2]
ic| sign_pos: (4, 1, 3, 2)
    solver.assertions(): [hut >= 1,
                          hut <= 4,
                          hut != 3,
                          Or(hut == 0, hut == 2),
                          Or(hut == 2, hut == 4),
                          And(hut != 1, hut != 3)]
    solver.model(): [hut = 2]
ic| sign_pos: (4, 3, 1, 2)
    solver.assertions(): 

## Using z3 to determine both signs' position `sign_pos` **and** solution `hut`

In [5]:
solver = z3.Solver()
hut = z3.Int("hut")
sign_pos = [z3.Int(f"sign_pos_{n}") for n in range(1, 4 + 1)]

solver.add(hut >= 1, hut <= 4)
for s in sign_pos:
    solver.add(s >= 1, s <= 4)
solver.add(z3.Distinct(sign_pos))

# Sign 1: IT IS NOT #3
solver.add(hut != 3)

# Sign 2: IT IS NEXT TO THIS ONE
solver.add(z3.Or(hut == sign_pos[1] - 1, hut == sign_pos[1] + 1))

# Sign 3: IT IS NEXT TO THIS ONE
solver.add(z3.Or(hut == sign_pos[2] - 1, hut == sign_pos[2] + 1))

# Sign 4: IT IS NOT NEXT TO THIS ONE
solver.add(z3.And(hut != sign_pos[3] - 1, hut != sign_pos[3] + 1))

if solver.check() == z3.sat:
    ic(solver.assertions(), solver.model())

ic| solver.assertions(): [hut >= 1,
                          hut <= 4,
                          sign_pos_1 >= 1,
                          sign_pos_1 <= 4,
                          sign_pos_2 >= 1,
                          sign_pos_2 <= 4,
                          sign_pos_3 >= 1,
                          sign_pos_3 <= 4,
                          sign_pos_4 >= 1,
                          sign_pos_4 <= 4,
                          Distinct(sign_pos_1, sign_pos_2, sign_pos_3, sign_pos_4),
                          hut != 3,
                          Or(hut == sign_pos_2 - 1, hut == sign_pos_2 + 1),
                          Or(hut == sign_pos_3 - 1, hut == sign_pos_3 + 1),
                          And(hut != sign_pos_4 - 1, hut != sign_pos_4 + 1)]
    solver.model(): [sign_pos_2 = 3,
                     hut = 2,
                     sign_pos_3 = 1,
                     sign_pos_1 = 4,
                     sign_pos_4 = 2]
